# FAST-UAV - Fixed-Wing Design Aerodynamics Partial check
*Author: Fernando Urquía Cutillas - 2026* <br>

This notebook checks the partials of the optimization groups through the built-in function check partials of OpenMDAO. It consists on comparing the relative error of the analytical method computed by the user in `def compute_partials()` against the finite difference computed with the data in `def compute()`

In [3]:
import openmdao.api as om
import numpy as np

from fastuav.models.aerodynamics.aerodynamics_fixedwing import WingParasiticDrag  # adjust path if needed

REL_TOL = 1e-5  # relaxed tolerance due to stdatm noise

# 1. Initialize
prob = om.Problem()
prob.model.add_subsystem("wing_drag", WingParasiticDrag(propulsion_id="fixedwing"), promotes=["*"])
prob.setup(force_alloc_complex=True)

# 2. Override the np.nan defaults
prob.set_val("data:geometry:wing:tc", 0.12)
prob.set_val("data:geometry:wing:MAC:length", 0.35, units="m")
prob.set_val("data:geometry:wing:surface", 0.8, units="m**2")
prob.set_val("mission:sizing:main_route:cruise:altitude", 150.0, units="m")
prob.set_val("mission:sizing:main_route:cruise:speed:fixedwing", 16.0, units="m/s")
prob.set_val("mission:sizing:dISA", 0.0, units="K")

# 3. Run once
prob.run_model()
print("CD0_wing =", prob.get_val("data:aerodynamics:CD0:wing"), "\n")

# 4. Check partials (central diff avoids forward-FD false-flags)
data = prob.check_partials(
    compact_print=True,
    show_only_incorrect=False,
    method="fd",
    form="central",
    step=1e-6,
)

# 5. Detailed results
print("\n" + "=" * 70)
print("DETAILED RESULTS  (rel-error threshold = %.0e)" % REL_TOL)
print("=" * 70 + "\n")

n_fail = 0
for comp_name in data:
    for (out, inp), err in data[comp_name].items():
        J_fwd = np.asarray(err["J_fwd"])
        J_fd  = np.asarray(err["J_fd"])
        rel_error = err["rel error"].forward
        abs_error = err["abs error"].forward

        if J_fwd.size == 1:
            a_str, fd_str = f"{J_fwd.item():12.6e}", f"{J_fd.item():12.6e}"
        else:
            a_str  = f"vector{J_fwd.shape} max|.|={np.max(np.abs(J_fwd)):.3e}"
            fd_str = f"vector{J_fd.shape} max|.|={np.max(np.abs(J_fd)):.3e}"

        is_zero = (np.max(np.abs(J_fwd)) == 0.0) and (np.max(np.abs(J_fd)) == 0.0)
        status = "PASS" if (is_zero or rel_error < REL_TOL) else "FAIL"
        n_fail += status == "FAIL"

        print(f"d({out}) / d({inp})")
        print(f"  Analytic:  {a_str}")
        print(f"  FD:        {fd_str}")
        print(f"  Rel Error: {rel_error:12.6e}")
        print(f"  Abs Error: {abs_error:12.6e}")
        print(f"  {status}\n")

print("=" * 70)
print("ALL PASSED" if n_fail == 0 else f"{n_fail} FAILED")
print("=" * 70)

CD0_wing = [0.01360354] 

----------------------------------------
Component: WingParasiticDrag 'wing_drag'
----------------------------------------

+------------------------------+----------------------------------------------------+-------------+-------------+-------------+-------------+------------+
| of '<variable>'              | wrt '<variable>'                                   |   calc mag. |  check mag. |  a(cal-chk) |  r(cal-chk) | error desc |
+==============================+====================================================+=============+=============+=============+=============+============+
| 'data:aerodynamics:CD0:wing' | 'data:geometry:wing:MAC:length'                    |  7.8068e-03 |  7.8068e-03 |  6.4868e-13 |  8.3093e-11 |            |
+------------------------------+----------------------------------------------------+-------------+-------------+-------------+-------------+------------+
| 'data:aerodynamics:CD0:wing' | 'data:geometry:wing:surface'              

In [4]:
import openmdao.api as om
import numpy as np

from fastuav.models.aerodynamics.aerodynamics_fixedwing import TailParasiticDrag  # adjust path

REL_TOL = 1e-4

# 1. Initialize (test horizontal tail)
prob = om.Problem()
prob.model.add_subsystem(
    "tail_drag",
    TailParasiticDrag(propulsion_id="fixedwing", tail="horizontal"),
    promotes=["*"]
)
prob.setup(force_alloc_complex=True)

# 2. Override defaults
prob.set_val("data:geometry:wing:tc", 0.12)
prob.set_val("data:geometry:wing:surface", 0.8, units="m**2")
prob.set_val("data:geometry:tail:horizontal:MAC:length", 0.25, units="m")
prob.set_val("data:geometry:tail:horizontal:surface", 0.25, units="m**2")
prob.set_val("mission:sizing:main_route:cruise:altitude", 150.0, units="m")
prob.set_val("mission:sizing:main_route:cruise:speed:fixedwing", 16.0, units="m/s")
prob.set_val("mission:sizing:dISA", 0.0, units="K")

# 3. Run
prob.run_model()
print("CD0_tail:horizontal =", prob.get_val("data:aerodynamics:CD0:tail:horizontal"), "\n")

# 4. Check partials
data = prob.check_partials(
    compact_print=True,
    show_only_incorrect=False,
    method="fd",
    form="central",
    step=1e-6,
)

# 5. Results
print("\n" + "=" * 70)
print("DETAILED RESULTS  (rel-error threshold = %.0e)" % REL_TOL)
print("=" * 70 + "\n")

n_fail = 0
for comp_name in data:
    for (out, inp), err in data[comp_name].items():
        J_fwd = np.asarray(err["J_fwd"])
        J_fd  = np.asarray(err["J_fd"])
        rel_error = err["rel error"].forward
        abs_error = err["abs error"].forward

        if J_fwd.size == 1:
            a_str, fd_str = f"{J_fwd.item():12.6e}", f"{J_fd.item():12.6e}"
        else:
            a_str  = f"vector{J_fwd.shape} max|.|={np.max(np.abs(J_fwd)):.3e}"
            fd_str = f"vector{J_fd.shape} max|.|={np.max(np.abs(J_fd)):.3e}"

        is_zero = (np.max(np.abs(J_fwd)) == 0.0) and (np.max(np.abs(J_fd)) == 0.0)
        status = "PASS" if (is_zero or rel_error < REL_TOL) else "FAIL"
        n_fail += status == "FAIL"

        print(f"d({out}) / d({inp})")
        print(f"  Analytic:  {a_str}")
        print(f"  FD:        {fd_str}")
        print(f"  Rel Error: {rel_error:12.6e}")
        print(f"  Abs Error: {abs_error:12.6e}")
        print(f"  {status}\n")

print("=" * 70)
print("ALL PASSED" if n_fail == 0 else f"{n_fail} FAILED")
print("=" * 70)

CD0_tail:horizontal = [0.00351956] 

----------------------------------------
Component: TailParasiticDrag 'tail_drag'
----------------------------------------

+-----------------------------------------+----------------------------------------------------+-------------+-------------+-------------+-------------+------------+
| of '<variable>'                         | wrt '<variable>'                                   |   calc mag. |  check mag. |  a(cal-chk) |  r(cal-chk) | error desc |
+=========================================+====================================================+=============+=============+=============+=============+============+
| 'data:aerodynamics:CD0:tail:horizontal' | 'data:geometry:tail:horizontal:MAC:length'         |  2.9038e-03 |  2.9038e-03 |  1.7556e-13 |  6.0459e-11 |            |
+-----------------------------------------+----------------------------------------------------+-------------+-------------+-------------+-------------+------------+
| 'data:a

In [6]:
import openmdao.api as om
import numpy as np

from fastuav.models.aerodynamics.aerodynamics_fixedwing import FuselageParasiticDrag  # adjust path

REL_TOL = 1e-4

# 1. Initialize
prob = om.Problem()
prob.model.add_subsystem(
    "fus_drag",
    FuselageParasiticDrag(propulsion_id="fixedwing"),
    promotes=["*"]
)
prob.setup(force_alloc_complex=True)

# 2. Override defaults
prob.set_val("data:geometry:fuselage:length", 1.3, units="m")
prob.set_val("data:geometry:fuselage:diameter:mid", 0.26, units="m")
prob.set_val("data:geometry:fuselage:fineness", 5.0)
prob.set_val("data:geometry:wing:surface", 0.8, units="m**2")
prob.set_val("mission:sizing:main_route:cruise:altitude", 150.0, units="m")
prob.set_val("mission:sizing:main_route:cruise:speed:fixedwing", 16.0, units="m/s")
prob.set_val("mission:sizing:dISA", 0.0, units="K")

# 3. Run
prob.run_model()
print("CD0_fuselage =", prob.get_val("data:aerodynamics:CD0:fuselage"), "\n")

# 4. Check partials
data = prob.check_partials(
    compact_print=True,
    show_only_incorrect=False,
    method="fd",
    form="central",
    step=1e-6,
)

# 5. Results
print("\n" + "=" * 70)
print("DETAILED RESULTS  (rel-error threshold = %.0e)" % REL_TOL)
print("=" * 70 + "\n")

n_fail = 0
for comp_name in data:
    for (out, inp), err in data[comp_name].items():
        J_fwd = np.asarray(err["J_fwd"])
        J_fd  = np.asarray(err["J_fd"])
        rel_error = err["rel error"].forward
        abs_error = err["abs error"].forward

        if J_fwd.size == 1:
            a_str, fd_str = f"{J_fwd.item():12.6e}", f"{J_fd.item():12.6e}"
        else:
            a_str  = f"vector{J_fwd.shape} max|.|={np.max(np.abs(J_fwd)):.3e}"
            fd_str = f"vector{J_fd.shape} max|.|={np.max(np.abs(J_fd)):.3e}"

        is_zero = (np.max(np.abs(J_fwd)) == 0.0) and (np.max(np.abs(J_fd)) == 0.0)
        status = "PASS" if (is_zero or rel_error < REL_TOL) else "FAIL"
        n_fail += status == "FAIL"

        print(f"d({out}) / d({inp})")
        print(f"  Analytic:  {a_str}")
        print(f"  FD:        {fd_str}")
        print(f"  Rel Error: {rel_error:12.6e}")
        print(f"  Abs Error: {abs_error:12.6e}")
        print(f"  {status}\n")

print("=" * 70)
print("ALL PASSED" if n_fail == 0 else f"{n_fail} FAILED")
print("=" * 70)

CD0_fuselage = [0.00615228] 

-------------------------------------------
Component: FuselageParasiticDrag 'fus_drag'
-------------------------------------------

+----------------------------------+----------------------------------------------------+-------------+-------------+-------------+-------------+------------+
| of '<variable>'                  | wrt '<variable>'                                   |   calc mag. |  check mag. |  a(cal-chk) |  r(cal-chk) | error desc |
+==================================+====================================================+=============+=============+=============+=============+============+
| 'data:aerodynamics:CD0:fuselage' | 'data:geometry:fuselage:diameter:mid'              |  2.3663e-02 |  2.3663e-02 |  8.7534e-13 |  3.6993e-11 |            |
+----------------------------------+----------------------------------------------------+-------------+-------------+-------------+-------------+------------+
| 'data:aerodynamics:CD0:fuselage' | 'data

In [7]:
import openmdao.api as om
import numpy as np

from fastuav.models.aerodynamics.components.compute_reynolds import ComputeUnitReynolds  # adjust path

REL_TOL = 1e-5

# Test CRUISE case
print("=" * 70)
print("TESTING CRUISE MODE")
print("=" * 70 + "\n")

prob = om.Problem()
prob.model.add_subsystem(
    "unit_re",
    ComputeUnitReynolds(low_speed_aero=False),
    promotes=["*"]
)
prob.setup(force_alloc_complex=True)

prob.set_val("mission:sizing:main_route:cruise:speed:fixedwing", 16.0, units="m/s")
prob.set_val("mission:sizing:main_route:cruise:altitude", 150.0, units="m")

prob.run_model()
print("Cruise Mach =", prob.get_val("data:aerodynamics:cruise:mach"))
print("Cruise unit_reynolds =", prob.get_val("data:aerodynamics:cruise:unit_reynolds"), "\n")

data = prob.check_partials(
    compact_print=True,
    show_only_incorrect=False,
    method="fd",
    form="central",
    step=1e-6,
)

print("\n" + "=" * 70)
print("CRUISE RESULTS  (rel-error threshold = %.0e)" % REL_TOL)
print("=" * 70 + "\n")

n_fail = 0
for comp_name in data:
    for (out, inp), err in data[comp_name].items():
        J_fwd = np.asarray(err["J_fwd"])
        J_fd  = np.asarray(err["J_fd"])
        rel_error = err["rel error"].forward
        abs_error = err["abs error"].forward

        if J_fwd.size == 1:
            a_str, fd_str = f"{J_fwd.item():12.6e}", f"{J_fd.item():12.6e}"
        else:
            a_str  = f"vector max|.|={np.max(np.abs(J_fwd)):.3e}"
            fd_str = f"vector max|.|={np.max(np.abs(J_fd)):.3e}"

        is_zero = (np.max(np.abs(J_fwd)) == 0.0) and (np.max(np.abs(J_fd)) == 0.0)
        status = "PASS" if (is_zero or rel_error < REL_TOL) else "FAIL"
        n_fail += status == "FAIL"

        print(f"d({out}) / d({inp})")
        print(f"  Analytic:  {a_str}")
        print(f"  FD:        {fd_str}")
        print(f"  Rel Error: {rel_error:12.6e}")
        print(f"  {status}\n")

print("=" * 70)
print("ALL PASSED" if n_fail == 0 else f"{n_fail} FAILED")
print("=" * 70)

# Test LOW-SPEED case
print("\n" + "=" * 70)
print("TESTING LOW-SPEED MODE")
print("=" * 70 + "\n")

prob2 = om.Problem()
prob2.model.add_subsystem(
    "unit_re_ls",
    ComputeUnitReynolds(low_speed_aero=True),
    promotes=["*"]
)
prob2.setup(force_alloc_complex=True)

prob2.set_val("data:TLAR:v_approach", 12.0, units="m/s")

prob2.run_model()
print("Low-speed Mach =", prob2.get_val("data:aerodynamics:low_speed:mach"))
print("Low-speed unit_reynolds =", prob2.get_val("data:aerodynamics:low_speed:unit_reynolds"), "\n")

data2 = prob2.check_partials(
    compact_print=True,
    show_only_incorrect=False,
    method="fd",
    form="central",
    step=1e-6,
)

print("\n" + "=" * 70)
print("LOW-SPEED RESULTS  (rel-error threshold = %.0e)" % REL_TOL)
print("=" * 70 + "\n")

n_fail2 = 0
for comp_name in data2:
    for (out, inp), err in data2[comp_name].items():
        J_fwd = np.asarray(err["J_fwd"])
        J_fd  = np.asarray(err["J_fd"])
        rel_error = err["rel error"].forward
        abs_error = err["abs error"].forward

        if J_fwd.size == 1:
            a_str, fd_str = f"{J_fwd.item():12.6e}", f"{J_fd.item():12.6e}"
        else:
            a_str  = f"vector max|.|={np.max(np.abs(J_fwd)):.3e}"
            fd_str = f"vector max|.|={np.max(np.abs(J_fd)):.3e}"

        is_zero = (np.max(np.abs(J_fwd)) == 0.0) and (np.max(np.abs(J_fd)) == 0.0)
        status = "PASS" if (is_zero or rel_error < REL_TOL) else "FAIL"
        n_fail2 += status == "FAIL"

        print(f"d({out}) / d({inp})")
        print(f"  Analytic:  {a_str}")
        print(f"  FD:        {fd_str}")
        print(f"  Rel Error: {rel_error:12.6e}")
        print(f"  {status}\n")

print("=" * 70)
print("ALL PASSED" if n_fail2 == 0 else f"{n_fail2} FAILED")
print("=" * 70)

TESTING CRUISE MODE

Cruise Mach = [0.04709774]
Cruise unit_reynolds = [1082492.78455353] 

----------------------------------------
Component: ComputeUnitReynolds 'unit_re'
----------------------------------------

+------------------------------------------+----------------------------------------------------+-------------+-------------+-------------+-------------+--------------------+
| of '<variable>'                          | wrt '<variable>'                                   |   calc mag. |  check mag. |  a(cal-chk) |  r(cal-chk) | error desc         |
+==========================================+====================================================+=============+=============+=============+=============+====================+
| 'data:aerodynamics:cruise:mach'          | 'mission:sizing:main_route:cruise:altitude'        |  5.3301e-07 |  5.3301e-07 |  2.8264e-12 |  5.3027e-06 |  >REL_TOL          |
+------------------------------------------+----------------------------------------

In [11]:
import openmdao.api as om
import numpy as np

from fastuav.models.aerodynamics.external.vlm.compute_aero import ComputeLocalReynolds  # adjust path

REL_TOL = 1e-5

# Test CRUISE case
print("=" * 70)
print("TESTING CRUISE MODE")
print("=" * 70 + "\n")

prob = om.Problem()
prob.model.add_subsystem(
    "local_re",
    ComputeLocalReynolds(low_speed_aero=False),
    promotes=["*"]
)
prob.setup(force_alloc_complex=True)

prob.set_val("data:aerodynamics:cruise:unit_reynolds", 1.5e6, units="m**-1")
prob.set_val("data:geometry:wing:MAC:length", 0.35, units="m")
prob.set_val("data:geometry:tail:horizontal:MAC:length", 0.25, units="m")

prob.run_model()
print("Wing Reynolds (cruise) =", prob.get_val("data:aerodynamics:wing:cruise:reynolds"))
print("HT Reynolds (cruise) =", prob.get_val("data:aerodynamics:horizontal_tail:cruise:reynolds"), "\n")

data = prob.check_partials(
    compact_print=True,
    show_only_incorrect=False,
    method="fd",
    form="central",
    step=1e-6,
)

print("\n" + "=" * 70)
print("CRUISE RESULTS  (rel-error threshold = %.0e)" % REL_TOL)
print("=" * 70 + "\n")

n_fail = 0
for comp_name in data:
    for (out, inp), err in data[comp_name].items():
        J_fwd = np.asarray(err["J_fwd"])
        J_fd  = np.asarray(err["J_fd"])
        rel_error = err["rel error"].forward
        abs_error = err["abs error"].forward

        if J_fwd.size == 1:
            a_str, fd_str = f"{J_fwd.item():12.6e}", f"{J_fd.item():12.6e}"
        else:
            a_str  = f"vector max|.|={np.max(np.abs(J_fwd)):.3e}"
            fd_str = f"vector max|.|={np.max(np.abs(J_fd)):.3e}"

        is_zero = (np.max(np.abs(J_fwd)) == 0.0) and (np.max(np.abs(J_fd)) == 0.0)
        status = "PASS" if (is_zero or rel_error < REL_TOL) else "FAIL"
        n_fail += status == "FAIL"

        print(f"d({out}) / d({inp})")
        print(f"  Analytic:  {a_str}")
        print(f"  FD:        {fd_str}")
        print(f"  Rel Error: {rel_error:12.6e}")
        print(f"  {status}\n")

print("=" * 70)
print("ALL PASSED" if n_fail == 0 else f"{n_fail} FAILED")
print("=" * 70)

# Test LOW-SPEED case
print("\n" + "=" * 70)
print("TESTING LOW-SPEED MODE")
print("=" * 70 + "\n")

prob2 = om.Problem()
prob2.model.add_subsystem(
    "local_re_ls",
    ComputeLocalReynolds(low_speed_aero=True),
    promotes=["*"]
)
prob2.setup(force_alloc_complex=True)

prob2.set_val("data:aerodynamics:low_speed:unit_reynolds", 1.2e6, units="m**-1")
prob2.set_val("data:geometry:wing:MAC:length", 0.35, units="m")
prob2.set_val("data:geometry:tail:horizontal:MAC:length", 0.25, units="m")

prob2.run_model()
print("Wing Reynolds (low-speed) =", prob2.get_val("data:aerodynamics:wing:low_speed:reynolds"))
print("HT Reynolds (low-speed) =", prob2.get_val("data:aerodynamics:horizontal_tail:low_speed:reynolds"), "\n")

data2 = prob2.check_partials(
    compact_print=True,
    show_only_incorrect=False,
    method="fd",
    form="central",
    step=1e-6,
)

print("\n" + "=" * 70)
print("LOW-SPEED RESULTS  (rel-error threshold = %.0e)" % REL_TOL)
print("=" * 70 + "\n")

n_fail2 = 0
for comp_name in data2:
    for (out, inp), err in data2[comp_name].items():
        J_fwd = np.asarray(err["J_fwd"])
        J_fd  = np.asarray(err["J_fd"])
        rel_error = err["rel error"].forward
        abs_error = err["abs error"].forward

        if J_fwd.size == 1:
            a_str, fd_str = f"{J_fwd.item():12.6e}", f"{J_fd.item():12.6e}"
        else:
            a_str  = f"vector max|.|={np.max(np.abs(J_fwd)):.3e}"
            fd_str = f"vector max|.|={np.max(np.abs(J_fd)):.3e}"

        is_zero = (np.max(np.abs(J_fwd)) == 0.0) and (np.max(np.abs(J_fd)) == 0.0)
        status = "PASS" if (is_zero or rel_error < REL_TOL) else "FAIL"
        n_fail2 += status == "FAIL"

        print(f"d({out}) / d({inp})")
        print(f"  Analytic:  {a_str}")
        print(f"  FD:        {fd_str}")
        print(f"  Rel Error: {rel_error:12.6e}")
        print(f"  {status}\n")

print("=" * 70)
print("ALL PASSED" if n_fail2 == 0 else f"{n_fail2} FAILED")
print("=" * 70)

TESTING CRUISE MODE

Wing Reynolds (cruise) = [525000.]
HT Reynolds (cruise) = [375000.] 

------------------------------------------
Component: ComputeLocalReynolds 'local_re'
------------------------------------------

+-----------------------------------------------------+--------------------------------------------+-------------+-------------+-------------+-------------+--------------------+
| of '<variable>'                                     | wrt '<variable>'                           |   calc mag. |  check mag. |  a(cal-chk) |  r(cal-chk) | error desc         |
+=====================================================+============================================+=============+=============+=============+=============+====================+
| 'data:aerodynamics:horizontal_tail:cruise:reynolds' | 'data:aerodynamics:cruise:unit_reynolds'   |  2.5000e-01 |  2.5000e-01 |  1.9036e-06 |  7.6144e-06 |  >ABS_TOL >REL_TOL |
+-----------------------------------------------------+------------